# Option 2: VGG19 Transfer Learning

This notebook builds an image classifier from the pretrained VGG19 network. The input images are resized to 150x150x3, the ImageNet convolutional layers are reused as a frozen feature extractor, and a new classification head is trained for the dataset classes.

In [ ]:
import matplotlib as mpl
import matplotlib._docstring as mpl_docstring

mpl._docstring = mpl_docstring

import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.optimizers import Adam
from sklearn.model_selection import train_test_split
import numpy as np
from pathlib import Path

In [3]:
image_size = (150, 150)
batch_size = 32
seed = 42

# Resolve the dataset path whether the notebook runs from the repo root or Assignment_2.
dataset_dir = Path("Assignment_2/ImageDataset/images")
if not dataset_dir.exists():
    dataset_dir = Path("ImageDataset/images")

image_dataset = tf.keras.utils.image_dataset_from_directory(
    str(dataset_dir),
    image_size=image_size,
    batch_size=batch_size,
    shuffle=True,
    seed=seed,
)

class_names = image_dataset.class_names
image_batches = []
label_batches = []

# Keep images and labels from the same batch so they stay correctly matched.
for images, labels in image_dataset:
    image_batches.append(images.numpy())
    label_batches.append(labels.numpy())

x_data = np.concatenate(image_batches, axis=0)
y_data = np.concatenate(label_batches, axis=0)

train_images, test_images, train_labels, test_labels = train_test_split(
    x_data,
    y_data,
    test_size=0.2,
    random_state=seed,
    stratify=y_data,
)

(x_train, y_train), (x_test, y_test) = (train_images, train_labels), (test_images, test_labels)

# VGG19 expects float32 RGB pixels preprocessed with the ImageNet channel means.
image_train = tf.keras.applications.vgg19.preprocess_input(x_train.astype("float32"))
image_test = tf.keras.applications.vgg19.preprocess_input(x_test.astype("float32"))

print(class_names)
print(image_train.shape, y_train.shape)
print(image_test.shape, y_test.shape)
print(image_train.dtype, image_train.min(), image_train.max())
print(image_test.dtype, image_test.min(), image_test.max())

Found 3000 files belonging to 3 classes.
['cats', 'dogs', 'panda']
(2400, 150, 150, 3) (2400,)
(600, 150, 150, 3) (600,)
float32 -123.68 151.061
float32 -123.68 151.061


In [4]:
num_classes = len(class_names)
input_shape = image_train.shape[1:]

# Pretrained VGG19 backbone - reuses ImageNet visual features.
base_model = tf.keras.applications.VGG19(
    input_shape=input_shape,
    include_top=False,
    weights="imagenet",
)
base_model.trainable = False

# Layer 1: Input layer - receives one VGG19-preprocessed RGB image.
inputs = Input(shape=input_shape, name="input_image")

# Layer 2: VGG19 feature extractor - frozen at first to prevent overfitting.
x = base_model(inputs, training=False)

# Layer 3: Global average pooling - converts feature maps into one feature vector.
x = tf.keras.layers.GlobalAveragePooling2D(name="global_average_pooling")(x)

# Layer 4: Dense layer - learns task-specific combinations of VGG19 features.
x = tf.keras.layers.Dense(256, activation="relu", name="dense_features")(x)

# Layer 5: Dropout layer - helps reduce overfitting during training.
x = tf.keras.layers.Dropout(0.4, name="dropout_regularization")(x)

# Layer 6: Output layer - predicts one probability for each dataset class.
outputs = tf.keras.layers.Dense(num_classes, activation="softmax", name="class_probabilities")(x)

model = Model(inputs=inputs, outputs=outputs, name="animal_vgg19_transfer")
model.compile(
    optimizer=Adam(learning_rate=0.0001),
    loss="sparse_categorical_crossentropy",
    metrics=["accuracy"],
)

model.summary()

80134624/80134624 ━━━━━━━━━━━━━━━━━━━━ 2s 0us/step


Model: "animal_vgg19_transfer"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ input_image (InputLayer)        │ (None, 150, 150, 3)    │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ vgg19 (Functional)              │ (None, 4, 4, 512)      │    20,024,384 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling          │ (None, 512)            │             0 │
│ (GlobalAveragePooling2D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_features (Dense)          │ (None, 256)            │       131,328 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_regularization          │ (None, 256)            │             0 │
│ (Dropout)                       │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ class_probabilities (Dense)     │ (None, 3)              │           771 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 20,156,483 (76.89 MB)

 Trainable params: 132,099 (516.01 KB)

 Non-trainable params: 20,024,384 (76.39 MB)

In [ ]:
callbacks = [
    tf.keras.callbacks.EarlyStopping(
        monitor="val_loss",
        patience=4,
        restore_best_weights=True,
    ),
    tf.keras.callbacks.ReduceLROnPlateau(
        monitor="val_loss",
        factor=0.5,
        patience=2,
        min_lr=1e-6,
    ),
]

history = model.fit(image_train, y_train, validation_data=(image_test, y_test), epochs=30, batch_size=batch_size, callbacks=callbacks, verbose=1, )

Epoch 1/30
21/75 ━━━━━━━━━━━━━━━━━━━━ 1:24 2s/step - accuracy: 0.4550 - loss: 9.2340

In [ ]:
test_loss, test_accuracy = model.evaluate(image_test, y_test, verbose=0)
print(f"Test loss: {test_loss:.4f}")
print(f"Test accuracy: {test_accuracy:.4f}")

In [ ]:
model_dir = Path("Assignment_2")
if not model_dir.exists():
    model_dir = Path(".")

model.save(model_dir / "animal_vgg19_transfer.keras")
model.save_weights(model_dir / "animal_vgg19_transfer.weights.h5")

model_json = model.to_json()
with open(model_dir / "animal_vgg19_transfer.json", "w") as json_file:
    json_file.write(model_json)

print(f"Saved model files to: {model_dir.resolve()}")